# Setup Completo para Proyecto Telco Churn

Esta notebook contiene el código para regenerar todos los archivos necesarios desde cero, evitando errores de compatibilidad.

In [3]:
# === SETUP COMPLETO: Ejecutar esta celda para regenerar TODO desde cero ===
# Esto evita errores de compatibilidad al recargar archivos antiguos.
# Ejecutar solo una vez al inicio, luego cargar desde .pkl para velocidad.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import numpy as np

# 1. Cargar y preparar datos
df = pd.read_csv('telco.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]
df[categorical_cols] = df[categorical_cols].astype('object')
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'No': 0, 'Yes': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Crear preprocessor con imputación
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SkPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)
preprocessor.fit(X_train)

# 3. Entrenar y guardar pipelines
# LR
pipe_lr = SkPipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])
pipe_lr.fit(X_train, y_train)
y_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]

# RF
pipe_rf = SkPipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42))])
pipe_rf.fit(X_train, y_train)
y_proba_rf = pipe_rf.predict_proba(X_test)[:, 1]

# XGBoost con scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
pipe_xgb = SkPipeline(steps=[('preprocessor', preprocessor), ('classifier', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, verbosity=0))])
pipe_xgb.fit(X_train, y_train)
y_proba_xgb = pipe_xgb.predict_proba(X_test)[:, 1]

# XGBoost + SMOTE
pipe_smote = ImbPipeline(steps=[('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('classifier', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42, verbosity=0))])
pipe_smote.fit(X_train, y_train)
y_proba_smote = pipe_smote.predict_proba(X_test)[:, 1]

# 4. Guardar todo
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(pipe_lr, 'pipeline_lr.pkl')
joblib.dump(pipe_rf, 'pipeline_rf.pkl')
joblib.dump(pipe_xgb, 'pipeline_xgb.pkl')
np.save('y_proba_lr.npy', y_proba_lr)
np.save('y_proba_rf.npy', y_proba_rf)
np.save('y_proba_xgb.npy', y_proba_xgb)
np.save('y_proba_smote.npy', y_proba_smote)

print("Setup completo: todos los archivos regenerados y guardados.")

Setup completo: todos los archivos regenerados y guardados.
